In [ ]:
input_folder = os.path.join("Data", "Geotiffs")

tif_files = [f for f in os.listdir(input_folder) if f.endswith('.tif')]

if not tif_files:
    raise FileNotFoundError(f"No .tif file found in {input_folder}. Please add your orthomosaic there.")

input_tiff = os.path.join(input_folder, tif_files[0])
output_dir = os.path.join("Data", "Tiles")

tile_index_csv = os.path.join(output_dir, "tile_index.csv")
tile_size = 1024


def tile_raster(input_tiff, output_dir, tile_size):
    os.makedirs(output_dir, exist_ok=True)

    with rasterio.open(input_tiff) as src:
        width, height = src.width, src.height
        meta = src.meta.copy()

        n_cols = math.ceil(width / tile_size)
        n_rows = math.ceil(height / tile_size)

        tile_index = []
        tile_counter = 0

        for i in range(n_cols):
            for j in range(n_rows):
                x_off = i * tile_size
                y_off = j * tile_size
                w = min(tile_size, width - x_off)
                h = min(tile_size, height - y_off)

                window = Window(x_off, y_off, w, h)
                transform = src.window_transform(window)

                meta.update({
                    "height": h,
                    "width": w,
                    "transform": transform
                })

                tile_name = f"tile_{tile_counter}.tif"
                tile_counter +=1
                tile_path = os.path.join(output_dir, tile_name)

                with rasterio.open(tile_path, 'w', **meta) as dst:
                    dst.write(src.read(window=window))

                # Save index info (for CSV next step)
                bounds = rasterio.windows.bounds(window, src.transform)
                tile_index.append([tile_name, *bounds])

        return tile_index


In [ ]:
#save tile index
def save_tile_index(tile_index, output_csv):
    df = pd.DataFrame(tile_index, columns=['tile', 'minx', 'miny', 'maxx', 'maxy'])
    df.to_csv(output_csv, index=False)
    
# Run the tiling
tiles = tile_raster(input_tiff, output_dir, tile_size)

# Save the index to CSV
df_index = pd.DataFrame(tiles, columns=['tile_name', 'west', 'south', 'east', 'north'])
df_index.to_csv(tile_index_csv, index=False)
print(f"Tiling complete. Index saved to {tile_index_csv}")

In [ ]:
#visualize tiles
def visualize_tiles_on_raster(input_tiff, tile_csv):
    with rasterio.open(input_tiff) as src:
        fig, ax = plt.subplots(figsize=(10, 10))
        rasterio.plot.show(src, ax=ax, title="Tile Grid QC")

        tiles = pd.read_csv(tile_csv)
        tile_boxes = gpd.GeoDataFrame(
            tiles,
            geometry=[box(row.minx, row.miny, row.maxx, row.maxy) for _, row in tiles.iterrows()],
            crs=src.crs
        )

        tile_boxes.boundary.plot(ax=ax, color='yellow', linewidth=1)
        plt.show()


In [ ]:
#run
if __name__ == "__main__":
    print("Tiling raster...")
    tile_index = tile_raster(input_tiff, output_dir, tile_size)
    save_tile_index(tile_index, tile_index_csv)

    print("Visualizing tile grid overlay...")
    visualize_tiles_on_raster(input_tiff, tile_index_csv)

    print("Done")